# Simple Linear regression model code from scratch

In [1]:
import pandas as pd
import numpy  as np

In [2]:
adv_df = pd.DataFrame(pd.read_csv(r'D:\9_Git_folder\advertising.csv'))
adv_df.head()

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,12.0
3,151.5,41.3,58.5,16.5
4,180.8,10.8,58.4,17.9


### Model code

In [3]:
class simple_linear_regression():
    
    def __init__(self,
                 interceptor: bool, 
                 scaling: bool,
                 df,
                 X_column: list[str],
                 y_column: list[str]
                ):

        self.interceptor = interceptor
        self.scaling = scaling
        self.df = df
        self.X_column = X_column
        self.y_column = y_column
        
    def train_test_split(self, 
                         train_size : float,
                         random_state: float = None):

        self.train_size = train_size
        
        if random_state != None:
            self.random_state = random_state
        else: self.random_state = np.random.seed()
                
        self.overall_column_list = self.X_column + self.y_column
        
        self.df = self.df[self.overall_column_list].reset_index()
        self.df.drop(columns = {'index'}, inplace = True)

        self.train_df = self.df.sample(frac = self.train_size, random_state = self.random_state)
        self.test_df  = self.df.drop(self.train_df.index)
        
        self.X_train = self.train_df[self.X_column]
        self.X_test  = self.test_df[self.X_column]
        self.y_train = self.train_df[self.y_column]
        self.y_test  = self.test_df[self.y_column]
        
        return self.X_train, self.X_test, self.y_train, self.y_test
        
    def fit(self, 
            X, 
            y, 
            epoch = 100, 
            m_initalize = None, 
            b_initialize = None, 
            learning_rate = 0.001):
        self.X = X
        self.y = y
        self.epoch = epoch
        self.m = m_initalize
        self.b = b_initialize
        self.lr = learning_rate
        
        self.X = np.array(self.X).reshape(self.X.shape[0], 1)
        self.y = np.array(self.y).reshape(self.y.shape[0], 1)
        
        if self.scaling == True:
            self.X = (self.X - np.min(self.X)) / (np.max(self.X) - np.min(self.X))

        if self.m == None:
            self.m = 0
        
        if self.b == None:
            self.b = 0
        
        self.m_list = []
        self.b_list = []
        self.loss_list = []
        
        for i in range(0,self.epoch):
            
            self.y_pred = self.m*self.X + self.b
            self.loss = np.mean((self.y - self.y_pred)**2)
            
            self.m_list.append(self.m)
            self.b_list.append(self.b)
            self.loss_list.append(self.loss)
            
            self.dl_db = (-2)*np.mean((self.y - self.y_pred))
            self.dl_dm = (-2)*(np.dot(self.X.T, (self.y - self.y_pred))/self.X.shape[0])
            
            self.b = self.b - self.lr*self.dl_db
            self.m = self.m - self.lr*self.dl_dm
        
        self.slr_df = pd.DataFrame({'m':self.m_list, 'b':self.b_list, 'loss':self.loss_list})
        
        self.optimal_m, self.optimal_b = tuple(self.slr_df[['m', 'b']][self.slr_df['loss'] == self.slr_df['loss'].min()].iloc[0])
        
        return self.optimal_m, self.optimal_b
    
    def prediction(self, X_test):
        
        self.X_test = X_test
        self.X_test = np.array(X_test).reshape(self.X_test.shape[0],1)

        if self.scaling == True:
            self.X_test = (self.X_test - np.min(self.X_test)) / (np.max(self.X_test) - np.min(self.X_test))

        self.y_test_pred = self.optimal_m*self.X_test + self.optimal_b
        
        return self.y_test_pred
    
    def MSE(self, X_test, y_test):
        
        self.X_test = X_test
        self.X_test = np.array(X_test).reshape(self.X_test.shape[0],1)

        if self.scaling == True:
            self.X_test = (self.X_test - np.min(self.X_test)) / (np.max(self.X_test) - np.min(self.X_test))
        
        self.y_test = y_test
        self.y_test = np.array(y_test).reshape(self.y_test.shape[0],1)
        
        self.y_test_pred = self.optimal_m*self.X_test + self.optimal_b
        
        self.MSE = np.mean((self.y_test - self.y_test_pred)**2)
        
        return self.MSE
    
    def score(self, X_test, y_test):
        
        self.X_test = X_test
        self.X_test = np.array(X_test).reshape(self.X_test.shape[0],1)
        
        if self.scaling == True:
            self.X_test = (self.X_test - np.min(self.X_test)) / (np.max(self.X_test) - np.min(self.X_test))

        self.y_test_pred = self.optimal_m*self.X_test + self.optimal_b
        self.y_test_mean = np.mean(self.y_test)
        
        self.MSE = np.mean((self.y_test - self.y_test_pred)**2)
        self.TSS = np.mean((self.y_test - self.y_test_mean)**2)
        
        self.score = 1 - (self.MSE / self.TSS)
        
        return self.score
    
    def optimal_m(self):
        
        return self.optimal_m
    
    def optimal_b(self):
        
        return self.optimal_b
    
    def summary(self):
        
        print("Here is the summary of the model:",'\n')
        print("The dataframe size:", self.df.shape[0])
        print("Number of X columns:", len(self.X_column))
        print("X columns name:", self.X_column)
        print("y columns name:", self.y_column)
        print("The optimal value for m:", self.optimal_m)
        print("The optimal value for b:", self.optimal_b)
        print("The r-square value:", self.score)
        
        

In [4]:
slr_model = simple_linear_regression(interceptor=True,
                                     scaling=True,
                                     df = adv_df,
                                     X_column = ['TV'],
                                     y_column = ['Sales'])

X_train, X_test, y_train, y_test = slr_model.train_test_split(train_size=0.8,random_state= 0)

slr_model.fit(X_train,y_train, m_initalize= 1, b_initialize= 1,epoch=100000)

slr_model.prediction(X_test)

slr_model.MSE(X_test, y_test)

slr_model.score(X_test, y_test)

slr_model.summary()

Here is the summary of the model: 

The dataframe size: 200
Number of X columns: 1
X columns name: ['TV']
y columns name: ['Sales']
The optimal value for m: [[16.6066546]]
The optimal value for b: 6.869281420151718
The r-square value: 0.7488702428943492
